# 06 — Semantic cache: answer repeated questions in a millisecond

Companion notebook to blog post **06 (Semantic cache)**. Half your users ask questions
someone already asked, in different words — serve them from memory: ~600× faster, $0.

The receptionist-who-remembers metaphor:
1. Visitors **repeat each other in different words** — exact-match caches catch none of it
2. She **matches meaning, not wording** — embed the QUESTION, cosine vs remembered questions
3. **"Close enough" needs a line** — the threshold; too loose serves WRONG answers
4. She changes the **economics, not the answers**

Needs an OpenAI API key (the misses run the full pipeline).

In [ ]:
%pip install -q fastembed numpy openai

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## Setup — the full pipeline the cache sits in front of (posts 00–05)

In [ ]:
import numpy as np
from fastembed import TextEmbedding
from openai import OpenAI

client = OpenAI()
emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

corpus = [
    "Grooming appointments must be booked at least 48 hours in advance",
    "The boarding facility closes at 7 pm on weekdays and 5 pm on weekends",
    "Dogs staying longer than three nights receive a complimentary bath before pickup",
    "Refunds for cancelled boarding are issued within 5 business days",
    "Bookings made for public holidays are non-refundable",
    "Refunds for cancelled grooming appointments are issued within 10 business days",
    "All pets must have up to date rabies vaccination records on file",
    "Daycare drop off starts at 6:30 am and the last pickup is at 8 pm",
    "A late pickup fee of 15 dollars applies for every 30 minutes after closing",
]
doc_embs = list(emb_model.embed(corpus))

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed(text):
    return list(emb_model.embed([text]))[0]

def rag_answer(question):
    q = embed(question)
    top = sorted(((cosine(q, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:3]
    ctx = "\n".join(f"[doc {i}] {corpus[i]}" for s, i in top)
    r = client.chat.completions.create(model="gpt-5.4-mini", temperature=0,
        messages=[{"role": "user", "content":
            f"Answer the question using ONLY the context below.\n\nContext:\n{ctx}\n\nQuestion: {question}\nAnswer:"}])
    return r.choices[0].message.content.strip()

## The cache from scratch — ~20 lines

Store (question, embedding, answer). Lookup = best cosine against everything
remembered, hit if ≥ threshold. Note the novelty: every previous post embedded
DOCUMENTS; the cache embeds QUESTIONS to find other questions.

In [ ]:
class SemanticCache:
    def __init__(self, threshold=0.80):
        self.threshold = threshold
        self.entries = []                      # (query_text, query_emb, answer)

    def lookup(self, question):
        if not self.entries:
            return None
        q = embed(question)
        score, best = max((cosine(q, e), (t, a)) for t, e, a in self.entries)
        if score >= self.threshold:
            return {"cached_q": best[0], "answer": best[1], "score": score}
        return None

    def store(self, question, answer):
        self.entries.append((question, embed(question), answer))

def cached_rag(question, cache):
    hit = cache.lookup(question)
    if hit:
        return hit["answer"], f"CACHE HIT (matched {hit['cached_q']!r} @ {hit['score']:.3f})"
    answer = rag_answer(question)
    cache.store(question, answer)
    return answer, "MISS -> full pipeline, stored"

## The hit — a paraphrase served in a millisecond

In [ ]:
import time

cache = SemanticCache(threshold=0.80)

q1 = "What time does the boarding facility close on weekends?"
t0 = time.perf_counter(); a1, how1 = cached_rag(q1, cache); t1 = time.perf_counter() - t0
print(f"Q1: {q1}\n    {how1}\n    A: {a1}\n    time: {t1:.2f} s")

q2 = "How late is boarding open on Saturdays?"
t0 = time.perf_counter(); a2, how2 = cached_rag(q2, cache); t2 = time.perf_counter() - t0
print(f"\nQ2: {q2}\n    {how2}\n    A: {a2}\n    time: {t2:.3f} s")

print(f"\nspeedup: {t1/t2:.0f}x")

## What the receptionist hears — similarity to the stored question

Good news: paraphrases high, unrelated questions low. Bad news: one TRUE paraphrase
scores 0.72 — a false MISS at threshold 0.80. False misses cost a pipeline run, not a
wrong answer. Remember that asymmetry.

In [ ]:
tests = [
    "Boarding closing time on Sunday?",
    "How late is boarding open on Saturdays?",
    "When does dog boarding shut on the weekend?",
    "What are the daycare drop off hours?",
    "How long do boarding refunds take?",
]
e1 = embed(q1)
for t in tests:
    print(f"  {cosine(e1, embed(t)):.3f}  {t}")

## The collision line — and the overlap no threshold can fix

Sweep the threshold over pairs that SHOULD hit (paraphrases) and pairs that MUST NOT
(different questions, different answers). Then look at the raw similarities sorted:
a MUST-NOT pair outscores two SHOULD pairs — the distributions OVERLAP. No threshold
catches those paraphrases without also serving wrong answers.

In [ ]:
SHOULD_HIT = [
    ("What time does the boarding facility close on weekends?", "How late is boarding open on Saturdays?"),
    ("How far in advance must grooming be booked?", "How early do I need to book a grooming appointment?"),
    ("What is the late pickup fee?", "How much do I pay if I pick up my dog late?"),
    ("Do long boarding stays include a free bath?", "Does my dog get a complimentary bath after a long stay?"),
]
MUST_NOT_HIT = [
    ("How long do boarding refunds take?", "How long do grooming refunds take?"),
    ("What time does the boarding facility close on weekends?", "What are the daycare drop off hours?"),
    ("What is the late pickup fee?", "How much does grooming cost?"),
    ("Do long boarding stays include a free bath?", "Does the grooming package include a bath?"),
]

print(f"{'threshold':>9} | {'good hits':>9} | {'wrong hits':>10}")
for th in [0.70, 0.75, 0.80, 0.85]:
    good = sum(cosine(embed(a), embed(b)) >= th for a, b in SHOULD_HIT)
    wrong = sum(cosine(embed(a), embed(b)) >= th for a, b in MUST_NOT_HIT)
    print(f"{th:>9} | {good:>7}/4 | {wrong:>8}/4")

print("\nraw pair similarities, sorted — find the overlap:")
pairs = [(cosine(embed(a), embed(b)), "SHOULD ", b) for a, b in SHOULD_HIT] + \
        [(cosine(embed(a), embed(b)), "MUSTN'T", b) for a, b in MUST_NOT_HIT]
for s, kind, b in sorted(pairs, reverse=True):
    print(f"  {s:.3f}  {kind}  {b[:55]}")

## The wrong answer, served — threshold 0.65, real output

A false hit isn't like an LLM hallucination: it repeats IDENTICALLY, for every user,
until evicted. When in doubt set the line HIGH and eat the false misses.

In [ ]:
danger = SemanticCache(threshold=0.65)
qa, qb = "How long do boarding refunds take?", "How long do grooming refunds take?"

a, how = cached_rag(qa, danger)
print(f"Q: {qa}\n    {how}\n    A: {a}")

b, how = cached_rag(qb, danger)
print(f"\nQ: {qb}\n    {how}\n    A: {b}")
print("    ^ WRONG — grooming refunds take 10 business days")

## PRODUCTION — the two war-story bugs, fixed in code

Course-scale results (post 05's pipeline underneath, 50 golden Qs warmed, 50 LLM
paraphrases fired): hit-rate 1.00 @ th 0.80 with ZERO wrong hits (property of that
topically-distinct question set, not a law!), latency 2.62→0.21 s (−92%),
faithfulness held. Two bugs worth coding around:

1. **Self-collapsing warm-up** — warming via the answer() path let questions hit each
   other's fresh entries; the "warmed" cache held ONE entry. Warm must store
   UNCONDITIONALLY.
2. **Self-polluting eval** — a cache that stores on miss mutates itself while you
   measure it. Evaluation needs a FROZEN flag: look up, never store.

Also: don't confuse with an *embedding cache* (caches text→vector calls), and tie
invalidation to corpus version — a cached answer outlives the document it came from.

In [ ]:
class ProductionCache(SemanticCache):
    def __init__(self, threshold=0.80):
        super().__init__(threshold)
        self.frozen = False                    # bug 2 fix: eval must not mutate state

    def warm(self, questions):
        for q in questions:                    # bug 1 fix: store unconditionally,
            self.store(q, rag_answer(q))       # never through the lookup path

    def answer(self, question):
        hit = self.lookup(question)
        if hit:
            return hit["answer"], "HIT"
        answer = rag_answer(question)
        if not self.frozen:
            self.store(question, answer)
        return answer, "MISS"

prod = ProductionCache(threshold=0.80)
prod.warm(["What time does the boarding facility close on weekends?",
           "How far in advance must grooming be booked?",
           "What is the late pickup fee?"])
print("entries after warm:", len(prod.entries), " (bug 1 would have left 1)")

prod.frozen = True                             # measuring now — no writes
for q in ["Boarding closing time on Sunday?",
          "How early do I need to book a grooming appointment?",
          "Do unused daycare days roll over?"]:
    a, how = prod.answer(q)
    print(f"  {how:4}  {q[:50]}  ->  {a[:50]}")
print("entries after frozen eval:", len(prod.entries), " (unchanged)")

## Recap — the receptionist's rules

1. **Visitors repeat each other** → embed the question, match by meaning
2. **~20 lines** → store (q, emb, answer); lookup = best cosine ≥ threshold; hits ~600× faster
3. **The line** → distributions can OVERLAP (must-not 0.698 > should 0.658); false miss =
   wasted run, false hit = repeating wrong answer → set high, measure YOUR traffic
4. **Economics, not answers** → −92% latency at course scale; warm unconditionally,
   freeze during eval, invalidate on corpus updates

**Next: agentic RAG (07)** — the finale: the pipeline becomes a loop with a brain.